# Exploração dos dados e avaliação visual dos modelos

Este notebook complementa o pipeline de ML do AquaFlow: documenta a exploração e preparação de uma série reproduzível e, em uma seção separada, mostra gráficos dos modelos usando os resultados do treinamento já executado.

Os dados são simulados. Os rótulos representam cenários criados pelo gerador, não vazamentos confirmados. O notebook não substitui nem altera o pipeline automatizado, a API, os alertas ou o dashboard.

In [ ]:
from datetime import datetime
from pathlib import Path
import hashlib
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import load
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve
from zoneinfo import ZoneInfo

current = Path.cwd().resolve()
root = next((p for p in (current, *current.parents) if (p / "ml" / "train_model.py").is_file()), None)
if root is None:
    raise RuntimeError("Abra o notebook dentro do repositório AquaFlow.")
sys.path.insert(0, str(root / "ml"))
from generate_mock_data import generate
from prepare_dataset import analyze, prepare

DATA_DIR = root / "data" / "ml" / "notebook"
RAW_PATH = DATA_DIR / "raw" / "mock_readings.csv"
PREPARED_DIR = DATA_DIR / "prepared"
FIGURES_DIR = DATA_DIR / "figures"
TRAINING_DIR = root / "data" / "ml" / "training"
TIMEZONE = "America/Sao_Paulo"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
pd.set_option("display.max_columns", 20)
print(f"Repositório: {root}")

## 1. Geração, análise e preparação

A série usa seed fixa, intervalo de cinco minutos e fuso local da propriedade. O pipeline valida registros, mantém features separadas dos rótulos e aplica as mesmas transformações determinísticas usadas no treino e na inferência.

In [ ]:
generate(RAW_PATH, seed=42, start=datetime.fromisoformat("2026-09-01T00:00:00-03:00"), days=16, interval_minutes=5, timezone=TIMEZONE)
quality_report = analyze(RAW_PATH, DATA_DIR / "analysis")
preparation_report = prepare(RAW_PATH, PREPARED_DIR, TIMEZONE)
raw = pd.read_csv(RAW_PATH)
features = pd.read_csv(PREPARED_DIR / "features.csv")
labels = pd.read_csv(PREPARED_DIR / "labels.csv")
metadata = pd.read_csv(PREPARED_DIR / "metadata.csv")
print(f"Qualidade: {quality_report['status']} | Leituras preparadas: {preparation_report['prepared_readings']}")
print(f"Features: {', '.join(features.columns)}")
print(f"Rótulos separados: {', '.join(labels.columns)}")
raw.head()

### Qualidade, cobertura e cenários

Intervalos sem telemetria entram na análise de cobertura, mas não viram amostras treináveis. IDs, serial, firmware, bateria e sinal são mantidos fora das features do classificador.

In [ ]:
quality_report

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
raw["scenario_label"].value_counts().sort_index().plot(kind="bar", color="#159db1", ax=ax)
ax.set(title="Intervalos gerados por cenário", xlabel="Cenário simulado", ylabel="Quantidade de intervalos")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "01-cenarios-gerados.png", bbox_inches="tight"); plt.show()

valid = raw.loc[raw["reading_present"] == 1].copy()
valid["flow_rate_liters_minute"] = pd.to_numeric(valid["flow_rate_liters_minute"])
fig, ax = plt.subplots(figsize=(9, 4))
for scenario, group in valid.groupby("scenario_label"):
    ax.hist(group["flow_rate_liters_minute"], bins=35, alpha=0.55, label=scenario)
ax.set(title="Distribuição da vazão por cenário", xlabel="Vazão (L/min)", ylabel="Leituras")
ax.legend(title="Cenário", frameon=False)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "02-vazao-por-cenario.png", bbox_inches="tight"); plt.show()
print(f"Cobertura: {quality_report['row_counts']['coverage_ratio']:.1%}")

### Transformações aplicadas

O volume acumulado vira variação entre leituras consecutivas válidas; o intervalo real também é preservado. Hora e dia da semana são codificados ciclicamente no fuso local. O scaler da regressão logística é ajustado dentro de cada treino, sem consultar o bloco de teste.

In [ ]:
features.describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, group in features.assign(anomaly_label=labels["anomaly_label"]).groupby("anomaly_label"):
    name = "Anômala (simulada)" if label else "Normal (simulada)"
    axes[0].hist(group["flow_rate_liters_minute"], bins=35, alpha=0.55, label=name)
    axes[1].hist(group["volume_delta_liters"], bins=35, alpha=0.55, label=name)
axes[0].set(title="Vazão preparada", xlabel="L/min", ylabel="Leituras")
axes[1].set(title="Variação de volume preparada", xlabel="Litros por intervalo", ylabel="Leituras")
for ax in axes: ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "03-features-por-classe.png", bbox_inches="tight"); plt.show()

## 2. Gráficos de avaliação dos modelos

Esta seção consome os artefatos do treinamento normal do Compose. Ela não treina um segundo modelo. Primeiro confere se a preparação deste notebook corresponde aos dados do relatório de treinamento; depois compara candidatos na validação e mede o artefato escolhido no bloco temporal de teste reservado.

In [ ]:
metrics_path = TRAINING_DIR / "metrics.json"
split_path = TRAINING_DIR / "split_manifest.json"
model_path = TRAINING_DIR / "best_model.joblib"
missing = [p for p in (metrics_path, split_path, model_path) if not p.is_file()]
if missing:
    raise FileNotFoundError("Artefatos de treinamento ausentes. Inicie docker compose up --build e execute novamente.")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
split_manifest = json.loads(split_path.read_text(encoding="utf-8"))
fingerprint = hashlib.sha256()
for path in (PREPARED_DIR / "features.csv", PREPARED_DIR / "labels.csv", PREPARED_DIR / "metadata.csv"):
    fingerprint.update(path.read_bytes())
if fingerprint.hexdigest() != metrics["training_data_sha256"]:
    raise ValueError("A série preparada não corresponde aos dados do modelo treinado.")
selected_name = metrics["selected_model"]
comparison = pd.DataFrame({name: {
    "Validação temporal": result["validation"]["average_precision"],
    "Rotina alterada": result["generalization_validation"]["average_precision"] if result.get("generalization_validation") else np.nan,
    "Pontuação de seleção": metrics["selection_scores"][name],
} for name, result in metrics["candidates"].items()}).T.sort_values("Pontuação de seleção", ascending=False)
comparison.rename_axis("Modelo").reset_index()

In [ ]:
ax = comparison[["Validação temporal", "Rotina alterada"]].plot(kind="bar", figsize=(10, 4), color=["#168ca1", "#e3a35f"])
ax.set(title="Average precision na validação", xlabel="Modelo", ylabel="Average precision", ylim=(0, 1))
ax.tick_params(axis="x", rotation=15); ax.legend(frameon=False)
fig = ax.get_figure(); fig.tight_layout(); fig.savefig(FIGURES_DIR / "04-comparacao-modelos.png", bbox_inches="tight"); plt.show()
print(f"Modelo selecionado: {selected_name}")

### Teste cronológico reservado

A matriz usa o limiar de 0,5 adotado pelo pipeline. Precisão-recall ajuda a observar a relação entre encontrar positivos e evitar falsos positivos; ROC/AUC resume a separação das classes em vários limiares. Métricas com dados simulados não provam desempenho em consumo real.

In [ ]:
test_start = datetime.fromisoformat(split_manifest["validation_end_local"]).astimezone(ZoneInfo("UTC"))
timestamps = pd.to_datetime(metadata["recorded_at"], utc=True)
test_mask = timestamps >= pd.Timestamp(test_start)
X_test = features.loc[test_mask].to_numpy(dtype=float)
y_test = labels.loc[test_mask, "anomaly_label"].to_numpy(dtype=int)
if len(y_test) == 0 or set(y_test) != {0, 1}:
    raise ValueError("A partição de teste precisa conter as duas classes.")
model = load(model_path)
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= metrics["threshold"]).astype(int)
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
pr_auc = average_precision_score(y_test, probabilities)
roc_auc = roc_auc_score(y_test, probabilities)
print(f"Amostras de teste: {len(y_test)} | AP: {pr_auc:.3f} | ROC AUC: {roc_auc:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(cm, cmap="Blues")
ax.set(title="Matriz de confusão — teste temporal", xlabel="Classe prevista", ylabel="Classe observada")
ax.set_xticks([0, 1], labels=["Normal", "Anômala"]); ax.set_yticks([0, 1], labels=["Normal", "Anômala"])
for row in range(2):
    for col in range(2): ax.text(col, row, str(cm[row, col]), ha="center", va="center", color="#10242b", fontsize=12)
fig.colorbar(image, ax=ax, label="Leituras"); fig.tight_layout()
fig.savefig(FIGURES_DIR / "05-matriz-confusao.png", bbox_inches="tight"); plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, probabilities)
fpr, tpr, _ = roc_curve(y_test, probabilities)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(recall, precision, color="#168ca1", linewidth=2)
axes[0].set(title=f"Precisão-recall (AP = {pr_auc:.3f})", xlabel="Recall", ylabel="Precisão", xlim=(0, 1), ylim=(0, 1.02))
axes[1].plot(fpr, tpr, color="#168ca1", linewidth=2, label=f"Modelo (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "--", color="#888888", label="Referência aleatória")
axes[1].set(title="Curva ROC no teste temporal", xlabel="Falsos positivos", ylabel="Verdadeiros positivos", xlim=(0, 1), ylim=(0, 1.02))
axes[1].legend(frameon=False); fig.tight_layout()
fig.savefig(FIGURES_DIR / "06-curvas-precisao-recall-roc.png", bbox_inches="tight"); plt.show()

## Interpretação e limitações

- A comparação usa validação temporal e perfil de rotina alterada; o teste final permanece separado da seleção.
- A matriz de confusão mostra erros no limiar atual; as curvas mostram como a avaliação muda com outros limiares.
- Todos os dados e rótulos são artificiais e definidos pelo gerador. Os gráficos avaliam o reconhecimento desses cenários, não vazamentos reais.
- As imagens são exibidas no notebook e salvas em data/ml/notebook/figures/, diretório local ignorado pelo Git.